# 07 — Interpretation

**Question:** what is the model relying on, and does it correspond to anything physically sensible?

For a health application, a model nobody can interrogate is hard to govern. Permutation importance is computed on held-out folds under the grouped split — impurity importance from a fitted forest is measured on training data and biased toward high-cardinality features, which is the confound this project is about.

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
from smartnet import config
from smartnet.data import loader
from smartnet.models.interpret import permutation_importances, block_importance, logistic_coefficients
from smartnet.visualization import plots as P

df = loader.load_analysis_frame()

# Computed here rather than loaded from disk, so the interpretability step is
# reproduced in front of the reader. ~15s: 48 features x 5 repeats x 2 folds.
imp = permutation_importances(df, 'motion_5cat', n_repeats=5, max_folds=2)
imp.head(12).round(4)

In [ ]:
fig = P.plot_feature_importance(imp); plt.show()

In [ ]:
block_importance(imp)

**The four window-level aggregates dominate; the 40 individual per-second lag features contribute almost nothing.** Mean z-axis displacement over the preceding 10 seconds is the strongest feature by nearly a factor of two.

This matches the published importance analysis, which found the top variables were averages over a single dimension across the two 10-second periods. Both point the same way: the discriminating information is the **aggregate orientation change of the net over a window**, not the fine structure within it.

Physically, that is coherent. Raising or lowering a net produces a sustained displacement; a body crossing the threshold produces a burst. Those are separable. The direction of the crossing is not.

## The interpretable model

In [ ]:
coef = logistic_coefficients(df, 'motion_5cat')
top = coef.abs().max(axis=1).sort_values(ascending=False).head(10).index
coef.loc[top].round(3)

## What these numbers are not

Permutation importance measures what a model **relies on**, not what **causes** a behaviour. These are associations between engineered signal features and a human-applied label.

`meanzover10vector_back` being important does not mean z-axis displacement causes someone to enter a net. It means that, in this dataset, with this sensor placement, that statistic helps separate labels a human assigned from video.

**Prediction ≠ causation.** Any operational use must rest on prospective validation, not on feature rankings.

## Deployment implication

The feature set could likely be cut from 48 columns to fewer than 10 with little loss — relevant for on-device or low-bandwidth inference in field conditions.

A calibrated model should also be allowed to **abstain**: output *net crossed* with high confidence, and decline to guess direction. Matching the output to the model's real capability is more useful than forcing a five-way decision it cannot support.